In [ ]:
import duckdb
p = r"..."
conn = duckdb.connect(p)

In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')


q = """
WITH filtered as (
    SELECT
        acct_num, 
        part_number,
        amount,
        year
    FROM sales s
    RIGHT JOIN customers c ON s.acct_num = c.child_acct
), 
grouped_2025 AS (
    SELECT
        acct_num, 
        part_number,
        SUM(amount) AS total_2025,
        year
    FROM filtered
    WHERE year = 2025
    GROUP BY acct_num, part_number, year
    ORDER BY acct_num
), grouped_2024 AS (
    SELECT
        acct_num, 
        part_number,
        SUM(amount) AS total_2024,
        year
    FROM filtered
    WHERE year = 2024
    GROUP BY acct_num, part_number, year
    ORDER BY acct_num
), ranked_2025 AS (
    SELECT 
        acct_num,
        part_number,
        total_2025,
        row_number() OVER (PARTITION BY acct_num ORDER BY total_2025 DESC) AS rank
    FROM grouped_2025
), rank_filtered_2025 AS (
SELECT 
    acct_num,
    part_number,
    total_2025,
    rank,
FROM ranked_2025
WHERE rank <= 10
)
SELECT 
    rf_25.acct_num,
    rf_25.part_number,
    rf_25.total_2025,
    g_24.total_2024,
    rf_25.rank
FROM rank_filtered_2025 rf_25
LEFT JOIN grouped_2024 g_24 ON rf_25.acct_num = g_24.acct_num AND rf_25.part_number = g_24.part_number
ORDER BY rf_25.acct_num, rf_25.rank 
"""

df = conn.query(query=q).df()

df

,acct_num,part_number,total_2025,total_2024,rank
0,ACC005644,KIPC2555B-3,8840.00,NaN,1
1,ACC005644,ST670,8242.66,399.16,2
2,ACC005644,PLCM-2-UNL,3105.78,1317.61,3
3,ACC005644,PA740,2784.00,480.00,4
4,ACC005644,PA762,1831.63,2652.72,5
...,...,...,...,...,...
269,WES931266,IM771PU,3031.90,NaN,2
270,WES931266,IBA3-W,2028.30,NaN,3
271,WES931266,IB14X14-W,325.04,NaN,4
272,WES931266,CO-PRJ-R,0.00,NaN,5
